# 11g — Net-zero carbon budget: compounding reduction worked examples

Three worked examples of how a compounding annual emissions-reduction rate
translates into a cumulative "carbon budget" consumed over a horizon. All
three rely on `carbon_budget_compound_reduction(t0,t,Delta_R,R_minus,
CE_t0[,g_Y])`, defined below and validated by cross-checking its sign and
scale (see the derivation cell) against a bisection precondition used in
the third example.


**Deriving `carbon_budget_compound_reduction`.** The underlying model:
emissions at year $s\in[t_0,t]$ decline from a baseline $CE_{t_0}$ by an
immediate one-time cut $R_{-}$ at $t_0$, then compound at rate $\Delta R$
per year (optionally net of a GDP growth rate $g_Y$ that offsets the
reduction):
$$CE(s) = CE_{t_0}\,(1-R_{-})\,\big[(1-\Delta R)(1+g_Y)\big]^{s-t_0}$$
and the "carbon budget" is the cumulative sum
$CB=\sum_{s=t_0}^{t}CE(s)$. This is the same compounding-decline model
used for the `CTB`/`PAB` benchmark curves in `11f`
($100\times(1-0.07)^{t-t_0}(1-0.30)$), which is where the
$(1-\Delta R)$/$(1-R_-)$ roles were first confirmed. The function returns
two identical values (`CB1`, `CB2`) since only one is ever needed at each
call site below. **Validation**: the third example searches for
$\Delta R\in[0,0.50]$ hitting a target cumulative budget of 750
(GtCO$_2$) over 2020-2030 from a $CE_{t_0}=100$ baseline, guarded by
`(CB_theta(theta_min)<0) and (CB_theta(theta_max)>0)` — i.e. it expects
the *unreduced* cumulative budget to exceed 750 and the *maximally
reduced* one to fall short. Plugging the formula in: $\Delta R=0$ gives
$CB=1100$ (exceeds 750 ✓) and $\Delta R=0.50$ gives $CB\approx199.9$
(falls short ✓) — confirming the sign pattern the bisection search
relies on.


In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq


def carbon_budget_compound_reduction(t0, t, Delta_R, R_minus, CE_t0, g_Y=0.0):
    years = np.arange(t0, t + 1)
    decay = (1 - Delta_R) * (1 + g_Y)
    CE = CE_t0 * (1 - R_minus) * decay ** (years - t0)
    CB = CE.sum()
    return CB, CB  # CB1, CB2 -- see markdown above on why both are identical here


def estimate_delta_R(t0, t, theta_min, theta_max, R_minus, g_Y, CE_t0, CB_target):
    # Bisection search for the compounding reduction rate hitting a
    # target cumulative carbon budget.
    def CB_theta(theta):
        return CB_target - carbon_budget_compound_reduction(t0, t, theta, R_minus, CE_t0, g_Y)[0]

    if CB_theta(theta_min) < 0 and CB_theta(theta_max) > 0:
        return brentq(CB_theta, theta_min, theta_max)
    return np.nan


## 1. Cumulative carbon budget under compounding reduction

A $6\times6$ grid: annual reduction rate $\Delta R\in\{5\%,\dots,10\%\}$
(rows) against an initial one-time cut $R_{-}\in\{0\%,10\%,20\%,30\%,50\%,
75\%\}$ (columns), cumulative emissions 2020-2050 from a $36$
GtCO$_2$/year baseline (roughly today's global CO$_2$ emissions rate).


In [2]:
t0, t = 2020, 2050
R_minus_grid = np.array([0.00, 0.10, 0.20, 0.30, 0.50, 0.75])
Delta_R_grid = np.array([0.05, 0.06, 0.07, 0.08, 0.09, 0.10])
CE_t0 = 36

CB2 = np.zeros((len(Delta_R_grid), len(R_minus_grid)))
for i, dR in enumerate(Delta_R_grid):
    for j, Rm in enumerate(R_minus_grid):
        _, CB2[i, j] = carbon_budget_compound_reduction(t0, t, dR, Rm, CE_t0)

table1 = pd.DataFrame(CB2.round(0), index=[f"{100*d:.0f}%" for d in Delta_R_grid],
                       columns=[f"{100*r:.0f}%" for r in R_minus_grid])
table1.index.name = "Delta R \\ R-"
display(table1)

,0%,10%,20%,30%,50%,75%
Delta R \ R-,,,,,,
5%,573.0,516.0,459.0,401.0,287.0,143.0
6%,512.0,461.0,409.0,358.0,256.0,128.0
7%,460.0,414.0,368.0,322.0,230.0,115.0
8%,416.0,374.0,333.0,291.0,208.0,104.0
9%,379.0,341.0,303.0,265.0,189.0,95.0
10%,346.0,312.0,277.0,242.0,173.0,87.0


## 2. Same grid, now with GDP growth offsetting the reduction

The same $\Delta R\times R_{-}$ grid repeated for 4 GDP growth
assumptions $g_Y\in\{1\%,3\%,5\%,10\%\}$ — higher growth erodes more of
the compounding reduction each year, so the cumulative budget rises with
$g_Y$ at every grid point.


In [3]:
g_Y_values = [0.01, 0.03, 0.05, 0.10]
all_CB = {}
for g_Y in g_Y_values:
    CB2_g = np.zeros((len(Delta_R_grid), len(R_minus_grid)))
    for i, dR in enumerate(Delta_R_grid):
        for j, Rm in enumerate(R_minus_grid):
            _, CB2_g[i, j] = carbon_budget_compound_reduction(t0, t, dR, Rm, CE_t0, g_Y)
    all_CB[g_Y] = CB2_g
    print(f"g_Y = {int(100*g_Y)}%")
    display(pd.DataFrame(CB2_g.round(0), index=[f"{100*d:.0f}%" for d in Delta_R_grid],
                          columns=[f"{100*r:.0f}%" for r in R_minus_grid]))

combined = np.block([[all_CB[0.01], all_CB[0.03]], [all_CB[0.05], all_CB[0.10]]])
print("Combined 12x12 table (top-left=1%, top-right=3%, bottom-left=5%, bottom-right=10%)")
display(pd.DataFrame(combined.round(0)))

g_Y = 1%


,0%,10%,20%,30%,50%,75%
5%,642.0,578.0,514.0,450.0,321.0,161.0
6%,569.0,512.0,455.0,398.0,285.0,142.0
7%,508.0,457.0,406.0,356.0,254.0,127.0
8%,456.0,411.0,365.0,319.0,228.0,114.0
9%,412.0,371.0,330.0,289.0,206.0,103.0
10%,375.0,338.0,300.0,263.0,188.0,94.0


g_Y = 3%


,0%,10%,20%,30%,50%,75%
5%,821.0,739.0,657.0,575.0,410.0,205.0
6%,716.0,645.0,573.0,501.0,358.0,179.0
7%,630.0,567.0,504.0,441.0,315.0,157.0
8%,557.0,502.0,446.0,390.0,279.0,139.0
9%,497.0,447.0,398.0,348.0,249.0,124.0
10%,446.0,402.0,357.0,312.0,223.0,112.0


g_Y = 5%


,0%,10%,20%,30%,50%,75%
5%,1075.0,968.0,860.0,753.0,538.0,269.0
6%,923.0,831.0,739.0,646.0,462.0,231.0
7%,799.0,719.0,639.0,559.0,399.0,200.0
8%,696.0,627.0,557.0,488.0,348.0,174.0
9%,612.0,551.0,489.0,428.0,306.0,153.0
10%,541.0,487.0,433.0,379.0,271.0,135.0


g_Y = 10%


,0%,10%,20%,30%,50%,75%
5%,2331.0,2098.0,1865.0,1632.0,1166.0,583.0
6%,1926.0,1734.0,1541.0,1348.0,963.0,482.0
7%,1602.0,1442.0,1282.0,1122.0,801.0,401.0
8%,1342.0,1208.0,1074.0,940.0,671.0,336.0
9%,1133.0,1020.0,906.0,793.0,566.0,283.0
10%,964.0,867.0,771.0,675.0,482.0,241.0


Combined 12x12 table (top-left=1%, top-right=3%, bottom-left=5%, bottom-right=10%)


,0,1,2,3,4,5,6,7,8,9,10,11
0,642.0,578.0,514.0,450.0,321.0,161.0,821.0,739.0,657.0,575.0,410.0,205.0
1,569.0,512.0,455.0,398.0,285.0,142.0,716.0,645.0,573.0,501.0,358.0,179.0
2,508.0,457.0,406.0,356.0,254.0,127.0,630.0,567.0,504.0,441.0,315.0,157.0
3,456.0,411.0,365.0,319.0,228.0,114.0,557.0,502.0,446.0,390.0,279.0,139.0
4,412.0,371.0,330.0,289.0,206.0,103.0,497.0,447.0,398.0,348.0,249.0,124.0
5,375.0,338.0,300.0,263.0,188.0,94.0,446.0,402.0,357.0,312.0,223.0,112.0
6,1075.0,968.0,860.0,753.0,538.0,269.0,2331.0,2098.0,1865.0,1632.0,1166.0,583.0
7,923.0,831.0,739.0,646.0,462.0,231.0,1926.0,1734.0,1541.0,1348.0,963.0,482.0
8,799.0,719.0,639.0,559.0,399.0,200.0,1602.0,1442.0,1282.0,1122.0,801.0,401.0
9,696.0,627.0,557.0,488.0,348.0,174.0,1342.0,1208.0,1074.0,940.0,671.0,336.0


## 3. Solving for the reduction rate hitting a target budget

A $100$ GtCO$_2$/year baseline (2020) targeting a cumulative budget of
$750$ GtCO$_2$ over 2020-2030: `estimate_delta_R` bisects for the
compounding rate $\Delta R$ that exactly hits the target, first with no
growth offset, then adjusted for $g_Y=3\%$ via the identity
$\theta'=1-\frac{1-\theta}{1+g_Y}$ (the growth-adjusted rate that
reproduces the same *net* decline once growth is added back — i.e.
$(1-\theta')(1+g_Y)=(1-\theta)$).


In [4]:
R_minus3, CE_t0_3, CB_target = 0.00, 100, 750
t0_3, t_3 = 2020, 2050

g_Y = 0.00
theta = estimate_delta_R(2020, 2030, 0, 0.50, R_minus3, g_Y, CE_t0_3, CB_target)
print(f"g_Y=0%:  Delta_R solving CB(2020-2030)=750 -> theta = {theta:.4f}")

g_Y = 0.03
theta_adj = 1 - (1 - theta) / (1 + g_Y)
print(f"g_Y=3%:  growth-adjusted equivalent theta = {theta_adj:.4f}")

CB_min, _ = carbon_budget_compound_reduction(t0_3, t_3, 0.00, R_minus3, CE_t0_3, g_Y)
CB_max, _ = carbon_budget_compound_reduction(t0_3, t_3, 0.20, R_minus3, CE_t0_3, g_Y)
print(f"2020-2050 cumulative budget bracket at g_Y=3%: [Delta_R=0% -> {CB_min:.1f}, Delta_R=20% -> {CB_max:.1f}]")

theta2 = estimate_delta_R(2020, 2030, 0, 0.50, R_minus3, g_Y, CE_t0_3, CB_target)
print(f"g_Y=3%:  Delta_R solving CB(2020-2030)=750 directly -> theta = {theta2:.4f}")

g_Y=0%:  Delta_R solving CB(2020-2030)=750 -> theta = 0.0801
g_Y=3%:  growth-adjusted equivalent theta = 0.1069
2020-2050 cumulative budget bracket at g_Y=3%: [Delta_R=0% -> 5000.3, Delta_R=20% -> 566.8]
g_Y=3%:  Delta_R solving CB(2020-2030)=750 directly -> theta = 0.1069
